In [86]:
import pandas as pd
import seaborn as sns
import urllib
import sqlalchemy

In [87]:
import psycopg2
import sys

In [88]:
from sqlalchemy import create_engine

In [89]:
db_params = {
    'dbname': 'guest',
    'user': 'guest',
    'password': '12345',
    'host': 'localhost',
    'port': '5432'
}

In [90]:
engine = create_engine(f'postgresql+psycopg2://{db_params['user']}:{db_params['password']}@{db_params["host"]}:{db_params['port']}/{db_params['dbname']}')
conn = engine.connect()

In [64]:
topic_by_abstracts = pd.read_csv('/Users/polinashcherbatiuk/study/dbms_25/Основы БД/topic_by_abstracts.csv')

In [65]:
topic_by_abstracts.drop(columns='Unnamed: 0')

,n_topic,topic_descrip,doi,title,abstract
0,1,1_firms_investors_financial_profitability,10.32609/0042-8736-2023-8-123-146,Russian households’ finances during the pandemic,This paper is based on data of the Household F...
1,1,1_firms_investors_financial_profitability,10.1109/ICDMW53433.2021.00139,Deep Reinforcement Learning Task for Portfolio...,This paper covers the analysis of contemporary...
2,1,1_firms_investors_financial_profitability,10.26794/2587-5671-2022-26-3-64-84,Endogeneity Problem in Corporate Finance: Theo...,Endogeneity can cause a significant bias in th...
3,1,1_firms_investors_financial_profitability,10.1080/1540496X.2018.1562897,"Puzzling Premiums on FX Markets: Carry Trade, ...",We construct and compare the results of exploi...
4,1,1_firms_investors_financial_profitability,10.22394/1993-7601-2022-65-65-76,Stock market and cryptocurrency market volatility,"In the last ten years, cryptocurrencies have d..."
...,...,...,...,...,...
6908,173,173_ussr_soviet_sovietrussian_stalinist,10.1080/00905992.2017.1335298,Restricting Russians: language and immigration...,"In 1956, a prominent faction within the leader..."
6909,173,173_ussr_soviet_sovietrussian_stalinist,10.1177/08883254221079799,Unsettling Borderlands: The Population Exchang...,The article examines the Soviet nationality po...
6910,173,173_ussr_soviet_sovietrussian_stalinist,10.1080/00905992.2015.1072811,Nationalism in the USSR: a historical and comp...,The late 1980s and early 1990s were characteri...
6911,173,173_ussr_soviet_sovietrussian_stalinist,10.1111/soc4.12641,The Russian elite's imperial nationalism and t...,Abstract This article uses various Russian eli...


In [66]:
full_upload = pd.read_csv('/Users/polinashcherbatiuk/study/dbms_25/Основы БД/full_upload.csv')

In [67]:
df = full_upload.join(topic_by_abstracts.set_index('doi'), on='doi', rsuffix='_topic')

In [68]:
keywords = pd.read_csv('/Users/polinashcherbatiuk/study/dbms_25/Основы БД/keywords.csv')

In [69]:
df1=df.join(keywords.set_index('doi'), on='doi', rsuffix='_keywords')

In [70]:
authors = pd.read_csv('/Users/polinashcherbatiuk/study/dbms_25/Основы БД/authors.csv')

In [71]:
all_df=df1.join(authors.set_index('doi'), on='doi', rsuffix='_auth')

In [72]:
fin = all_df.drop(columns=['Unnamed: 0', 'Unnamed: 0_keywords', 'title_keywords', 'Unnamed: 0_auth', 'title_auth', 'Unnamed: 0_topic', 'abstract_topic', 'title_topic', 'creator'])

In [73]:
column_names = ['subtypeDescription', 'subjareas', 'main_author', 'secondary_authors', 'doi', 'title', 'publicationName', 'volume',
       'pageRange', 'openaccess', 'affiliation_country',
       'affiliation_city', 'affilname', 'OpenAlex_topconcepts', 'abstract',
       'n_topic', 'topic_descrip', 'keywords']

In [74]:
fin = fin.reindex(columns=column_names)

In [75]:
fin.columns

Index(['subtypeDescription', 'subjareas', 'main_author', 'secondary_authors',
       'doi', 'title', 'publicationName', 'volume', 'pageRange', 'openaccess',
       'affiliation_country', 'affiliation_city', 'affilname',
       'OpenAlex_topconcepts', 'abstract', 'n_topic', 'topic_descrip',
       'keywords'],
      dtype='object')

In [76]:
fin.set_index('doi', inplace=True)

In [77]:
import re

def standardize_author(name):
    # Обработка пустых значений
    if pd.isna(name) or name is None or name.strip() == '':
        return name
    
    # Обработка специальных меток для одиночных авторов
    if name.lower() in ["single author", "no info"]:
        return "Single Author"
    
    # Обработка формата "Фамилия И.О." или "Фамилия И. О."
    if re.match(r'^[A-ZА-Я][a-zа-я]+\s+[A-ZА-Я]\.\s*[A-ZА-Я]\.$', name):
        return name
    
    # Обработка формата "ФАМИЛИЯ И.О." (все в верхнем регистре)
    if re.match(r'^[A-ZА-Я]+\s+[A-ZА-Я]\.[A-ZА-Я]\.$', name):
        surname = name.split()[0].title()
        initials = name.split()[1]
        return f"{surname} {initials[0]}.{initials[2]}."
    
    # Обработка формата "Имя Отчество Фамилия" 
    parts = name.split()
    if len(parts) == 3:
        first, middle, last = parts
        first_initial = first[0]
        middle_initial = middle[0]
        return f"{last} {first_initial}. {middle_initial}."
    
    # Обработка формата "Имя Фамилия"
    elif len(parts) == 2 and not any(c in parts[1] for c in "."):
        first, last = parts
        return f"{last} {first[0]}."
    
    # Обработка формата "Фамилия O."
    if re.match(r'^[A-ZА-Я][a-zа-я]+\s+[A-ZА-Я]\.$', name):
        parts = name.split()
        return f"{parts[0]} {parts[1]}"
    
    # Возвращаем исходное имя, если формат не распознан
    return name

# Функция для обработки DataFrame
def process_main_author(df):
    # Создаем копию DataFrame, чтобы избежать предупреждений
    result_df = df.copy()
    
    # Стандартизация основного автора
    result_df['main_author'] = result_df['main_author'].apply(standardize_author)
    
    return result_df

processed_df = process_main_author(fin)

In [78]:
def split_and_standardize_secondary_authors(authors_string):
    # Проверка на NaN/None значения
    if pd.isna(authors_string) or authors_string is None or authors_string == '':
        return []
    
    # Разделяем строку по запятой
    authors_list = [author.strip() for author in authors_string.split(',')]
    
    # Применяем функцию стандартизации к каждому автору
    standardized_authors = [standardize_author(author) for author in authors_list]
    
    # Фильтруем "Single Author" и пустые значения
    standardized_authors = [author for author in standardized_authors 
                           if author != "Single Author" and author.strip() != '']
    
    return standardized_authors


# Функция для обработки DataFrame
def process_secondary_authors(df):
    # Создаем копию DataFrame, чтобы избежать предупреждений
    result_df = df.copy()
    
    # Преобразуем secondary_authors из текстовых строк в списки стандартизированных имен
    result_df['secondary_authors_list'] = result_df['secondary_authors'].apply(split_and_standardize_secondary_authors)
    
    # Если вам нужно также сохранить secondary_authors как текстовую строку, но уже стандартизированную:
    result_df['secondary_authors_standardized'] = result_df['secondary_authors_list'].apply(
        lambda authors: ', '.join(authors) if authors else None
    )
    
    return result_df


processed_df = process_secondary_authors(processed_df)

In [79]:
processed_df['secondary_authors'] = processed_df['secondary_authors_standardized']
processed_df = processed_df.drop(['secondary_authors_list', 'secondary_authors_standardized'], axis=1)

In [80]:
processed_df

,subtypeDescription,subjareas,main_author,secondary_authors,title,publicationName,volume,pageRange,openaccess,affiliation_country,affiliation_city,affilname,OpenAlex_topconcepts,abstract,n_topic,topic_descrip,keywords
doi,,,,,,,,,,,,,,,,,
10.1007/s11185-024-09307-1,Article,"SOCI, ARTS, PSYC",Letuchiy A.,None,Interpretation of complex sentences with diffe...,Russian Linguistics,49,NaN,0,Russian Federation;Russian Federation,Moscow;Moscow,V.V. Vinogradov Russian Language Institute of ...,"Negation,Interpretation (philosophy),Linguisti...",NaN,NaN,NaN,"modal operators, matrix predicates, different ..."
10.20900/jsr20250002,Article,"SOCI, ENVI, ENER",Kilinc-Ata N.,None,The Impact of Uncertainty in Economic Policy o...,Journal of Sustainability Research,7,NaN,1,Oman;Russian Federation,Muscat;Moscow,Sultan Qaboos University;HSE University,"China,Economics",NaN,NaN,NaN,"load capacity factors, economic policy, uncert..."
10.1016/j.wsif.2024.103049,Article,SOCI,Shmidova E.,Mikhaylova O.,Gendered perspectives on obsessive-compulsive ...,Women's Studies International Forum,109,NaN,0,Russian Federation,Moscow,HSE University,Obsessive compulsive,NaN,NaN,NaN,"clinically diagnosed individuals, self identif..."
10.4324/9781003262053-5,Book Chapter,"SOCI, ARTS, ECON",Chesnokova N.A.,None,"Korea as ""Little China"" (So Chunghwa)",The Routledge Handbook of Early Modern Korea,NaN,43-57,0,Russian Federation,Moscow,HSE University,China,Focusing on the ideological dimensions of Chos...,NaN,NaN,"little china, korea, chunghwa"
10.1108/ILS-10-2023-0137,Article,"SOCI, COMP",Koroleva D.,Jogezai N. A.,"The desire path: unleashing expectations, disc...",Information and Learning Science,126,110-131,0,Russian Federation,Moscow,HSE University,NaN,Purpose The purpose of this study is to demons...,10.0,10_pedagogical_educational_teaching_classroom,"way forward, higher education, gai use, proposing"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
10.1055/s-0036-1572383,Article,"HEAL, NURS",Malyutina S.,"Richardson J. D., Ouden D. d.",Verb Argument Structure in Narrative Speech: M...,Seminars in Speech and Language,37,34-47,0,Russian Federation;United States,Moscow;Columbia,HSE University;University of South Carolina,"Verb,Argument (complex analysis),Aphasia,Subca...",Previous research has found that verb argument...,103.0,103_aphasia_impairment_impairments_cognitive,verb argument structure
10.1177/17407745211008540,Article,PHAR,Chirkova A.,"Petrenko A., Vasilyev P.",Testing Meldonium: Assessing Soviet pragmatic ...,Clinical Trials,18,269-276,0,Russian Federation,Moscow,HSE University,Documentation,Background/aims Current research largely tends...,159.0,159_medicine_soviet_pharmaceutical_ussr,randomized controlled trial
10.1097/JNC.0000000000000344,Article,NURS,Meylakhs P.,"Davitadze A., Meylakhs A., Rodionova T., Aliev...",HIV Testing Patterns Among Recently Self-Teste...,Journal of the Association of Nurses in AIDS Care,33,550-558,0,Russian Federation,Moscow,HSE University,Men who have sex with men,Abstract Most qualitative research to date on ...,NaN,NaN,"men, qualitative study, sex"


In [23]:
processed_df.to_sql(name = 'full_database_normalised', con=conn, index = True)

420